# Session 7: Building LLM Agents

## Objectives
- Understand the agent architecture (Observe â†’ Think â†’ Act loop)
- Build an autonomous agent that uses tools iteratively
- Implement conversation memory
- Handle multi-step reasoning tasks

**Duration:** 40 minutes | **Level:** Medium

**Key difference from Session 6:** In function calling, the LLM makes ONE tool call and responds. An **agent** loops â€” it can call tools repeatedly until the task is done.

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from .env file
load_dotenv(dotenv_path=os.path.join("..", ".env"))

client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

print(f"Setup complete! Model: {MODEL}")

Setup complete! Model: gpt-5-mini


## 1. What Is an Agent?

An **agent** is an LLM that can:
- **Observe**: Read the current state and tool results
- **Think**: Decide what to do next
- **Act**: Call a tool or provide a final answer
- **Loop**: Repeat until the task is complete

```
User Task â†’ LLM thinks â†’ Calls Tool A â†’ Gets Result â†’ LLM thinks again â†’ Calls Tool B â†’ Gets Result â†’ LLM gives Final Answer
```

This is the **ReAct** pattern: **Re**ason + **Act** in a loop.

## 2. Define Agent Tools

Let's give our agent a set of useful tools.

In [2]:
import math

# Tool implementations
def calculator(expression):
    """Evaluate a math expression safely."""
    allowed_chars = set('0123456789+-*/.() ')
    if not all(c in allowed_chars for c in expression):
        return json.dumps({"error": "Invalid characters in expression"})
    try:
        result = eval(expression)
        return json.dumps({"expression": expression, "result": round(result, 6)})
    except Exception as e:
        return json.dumps({"error": str(e)})

def search_knowledge(query):
    """Simulate searching a knowledge base."""
    knowledge = {
        "population": {"France": "68 million", "Japan": "125 million", "Brazil": "214 million", "Germany": "84 million"},
        "capital": {"France": "Paris", "Japan": "Tokyo", "Brazil": "Brasilia", "Germany": "Berlin"},
        "gdp": {"France": "$2.78 trillion", "Japan": "$4.23 trillion", "Brazil": "$1.92 trillion", "Germany": "$4.07 trillion"},
        "area_km2": {"France": 643801, "Japan": 377975, "Brazil": 8515767, "Germany": 357022}
    }
    results = []
    query_lower = query.lower()
    for category, data in knowledge.items():
        for country, value in data.items():
            if country.lower() in query_lower or category in query_lower:
                results.append({"country": country, "category": category, "value": value})
    return json.dumps(results if results else [{"message": "No results found"}])

def get_exchange_rate(from_currency, to_currency):
    """Get exchange rate (simulated)."""
    rates = {
        ("USD", "EUR"): 0.92, ("EUR", "USD"): 1.09,
        ("USD", "JPY"): 149.50, ("JPY", "USD"): 0.0067,
        ("USD", "GBP"): 0.79, ("GBP", "USD"): 1.27,
        ("EUR", "GBP"): 0.86, ("GBP", "EUR"): 1.16,
    }
    rate = rates.get((from_currency.upper(), to_currency.upper()))
    if rate:
        return json.dumps({"from": from_currency, "to": to_currency, "rate": rate})
    return json.dumps({"error": f"Rate not found for {from_currency} to {to_currency}"})

# Tool schemas
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Calculate a mathematical expression. Supports +, -, *, /, parentheses.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Math expression to evaluate"}
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_knowledge",
            "description": "Search a knowledge base for country information (population, capital, GDP, area).",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query about countries"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_exchange_rate",
            "description": "Get the exchange rate between two currencies.",
            "parameters": {
                "type": "object",
                "properties": {
                    "from_currency": {"type": "string", "description": "Source currency code (e.g., USD)"},
                    "to_currency": {"type": "string", "description": "Target currency code (e.g., EUR)"}
                },
                "required": ["from_currency", "to_currency"]
            }
        }
    }
]

available_functions = {
    "calculator": calculator,
    "search_knowledge": search_knowledge,
    "get_exchange_rate": get_exchange_rate
}

print(f"Agent tools: {list(available_functions.keys())}")

Agent tools: ['calculator', 'search_knowledge', 'get_exchange_rate']


## 3. The Agent Loop

The core of an agent: keep calling the LLM until it gives a final answer (no more tool calls).

In [3]:
def run_agent(user_message, max_iterations=5, verbose=True):
    """Run an agent loop that can use tools iteratively."""
    
    system_prompt = """You are a helpful research assistant with access to tools.
Use the available tools to find information and perform calculations.
Think step by step. Break complex questions into smaller parts.
When you have enough information, provide a clear final answer."""
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message}
    ]
    
    for i in range(max_iterations):
        if verbose:
            print(f"\n--- Agent Step {i+1} ---")
        
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )
        
        message = response.choices[0].message
        
        # If no tool calls, the agent is done
        if not message.tool_calls:
            if verbose:
                print("Agent finished (no more tool calls)")
            return message.content
        
        # Process tool calls
        messages.append(message)
        
        for tool_call in message.tool_calls:
            func_name = tool_call.function.name
            func_args = json.loads(tool_call.function.arguments)
            
            if verbose:
                print(f"  Tool: {func_name}({func_args})")
            
            # Execute the function
            result = available_functions[func_name](**func_args)
            
            if verbose:
                print(f"  Result: {result}")
            
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })
    
    return "Agent reached maximum iterations without a final answer."

print("Agent function defined!")

Agent function defined!


In [4]:
# Simple task â€” requires one tool call
result = run_agent("What is the population of France?")
print(f"\nFinal Answer: {result}")


--- Agent Step 1 ---


  Tool: search_knowledge({'query': 'population of France'})
  Result: [{"country": "France", "category": "population", "value": "68 million"}, {"country": "Japan", "category": "population", "value": "125 million"}, {"country": "Brazil", "category": "population", "value": "214 million"}, {"country": "Germany", "category": "population", "value": "84 million"}, {"country": "France", "category": "capital", "value": "Paris"}, {"country": "France", "category": "gdp", "value": "$2.78 trillion"}, {"country": "France", "category": "area_km2", "value": 643801}]

--- Agent Step 2 ---


Agent finished (no more tool calls)

Final Answer: The population of France is about 68 million people.


In [5]:
# Complex task â€” requires MULTIPLE tool calls and reasoning
result = run_agent(
    "What is the population density of Japan? (population divided by area in km2)"
)
print(f"\nFinal Answer: {result}")


--- Agent Step 1 ---


  Tool: search_knowledge({'query': 'Japan population and area'})
  Result: [{"country": "France", "category": "population", "value": "68 million"}, {"country": "Japan", "category": "population", "value": "125 million"}, {"country": "Brazil", "category": "population", "value": "214 million"}, {"country": "Germany", "category": "population", "value": "84 million"}, {"country": "Japan", "category": "capital", "value": "Tokyo"}, {"country": "Japan", "category": "gdp", "value": "$4.23 trillion"}, {"country": "Japan", "category": "area_km2", "value": 377975}]

--- Agent Step 2 ---


  Tool: calculator({'expression': '125000000 / 377975'})
  Result: {"expression": "125000000 / 377975", "result": 330.709703}

--- Agent Step 3 ---


Agent finished (no more tool calls)

Final Answer: Population: 125,000,000; area: 377,975 km\^2.

Calculation:
$$\text{density} = \frac{125000000}{377975} \approx 330.71\ \text{people/km}^2$$

Answer: ≈ 330.7 people per km².


In [6]:
# Even more complex â€” multi-step with different tools
result = run_agent(
    "If France's GDP is divided equally among its population, how much would each person get in EUR?"
)
print(f"\nFinal Answer: {result}")


--- Agent Step 1 ---


  Tool: search_knowledge({'query': 'France GDP and population latest value GDP (nominal) and population'})
  Result: [{"country": "France", "category": "population", "value": "68 million"}, {"country": "Japan", "category": "population", "value": "125 million"}, {"country": "Brazil", "category": "population", "value": "214 million"}, {"country": "Germany", "category": "population", "value": "84 million"}, {"country": "France", "category": "capital", "value": "Paris"}, {"country": "France", "category": "gdp", "value": "$2.78 trillion"}, {"country": "Japan", "category": "gdp", "value": "$4.23 trillion"}, {"country": "Brazil", "category": "gdp", "value": "$1.92 trillion"}, {"country": "Germany", "category": "gdp", "value": "$4.07 trillion"}, {"country": "France", "category": "area_km2", "value": 643801}]

--- Agent Step 2 ---


  Tool: get_exchange_rate({'from_currency': 'USD', 'to_currency': 'EUR'})
  Result: {"from": "USD", "to": "EUR", "rate": 0.92}

--- Agent Step 3 ---


Agent finished (no more tool calls)

Final Answer: - Data used: GDP = \$2.78 trillion, population = 68 million, USD→EUR = 0.92.

Calculation (Python):
```python
# Python
gdp_usd = 2.78e12
pop = 68e6
rate = 0.92
per_person_eur = gdp_usd * rate / pop
per_person_eur
```

Math:
$$\text{per person}=\frac{2.78\times10^{12}\times0.92}{68\times10^{6}} \approx 37{,}609\ \text{EUR}.$$

Answer: about €37,609 per person.


## 4. Agent with Conversation Memory

Let's build an agent that remembers previous messages in a conversation.

In [7]:
class ConversationalAgent:
    """An agent with conversation memory."""
    
    def __init__(self, system_prompt=None):
        self.system_prompt = system_prompt or (
            "You are a helpful research assistant. Use tools when needed. "
            "Remember context from earlier in the conversation."
        )
        self.conversation_history = [
            {"role": "system", "content": self.system_prompt}
        ]
    
    def chat(self, user_message, max_iterations=5):
        """Send a message and get a response, maintaining history."""
        self.conversation_history.append({"role": "user", "content": user_message})
        
        # Create a working copy of messages for tool call processing
        messages = self.conversation_history.copy()
        
        for _ in range(max_iterations):
            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                tools=tools
            )
            
            message = response.choices[0].message
            
            if not message.tool_calls:
                # Add assistant's final response to history
                self.conversation_history.append(
                    {"role": "assistant", "content": message.content}
                )
                return message.content
            
            # Process tool calls
            messages.append(message)
            for tool_call in message.tool_calls:
                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments)
                result = available_functions[func_name](**func_args)
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result
                })
        
        return "Max iterations reached."

print("ConversationalAgent defined!")

ConversationalAgent defined!


In [8]:
# Test conversation memory
agent = ConversationalAgent()

# Turn 1
print("User: What's the population of Germany?")
print(f"Agent: {agent.chat('What is the population of Germany?')}")

print()

# Turn 2 â€” references "that country" from turn 1
print("User: What's the GDP of that country?")
print(f"Agent: {agent.chat('What about the GDP of that country?')}")

print()

# Turn 3 â€” references both previous answers
print("User: Calculate GDP per capita.")
print(f"Agent: {agent.chat('Now calculate the GDP per capita from those numbers.')}")

User: What's the population of Germany?


Agent: The population of Germany is about 84 million people.

User: What's the GDP of that country?


Agent: Germany's nominal GDP is about $4.07 trillion (approx.).

User: Calculate GDP per capita.


Agent: $$\text{GDP per capita}=\frac{4.07\times10^{12}}{84\times10^{6}}\approx 48{,}452\ \text{USD}$$


## Exercise: Extend the Agent

The agent loop pattern is the same for any tools.
Try asking complex multi-step questions!

In [9]:
# Try your own complex queries!
result = run_agent(
    "Compare the population density of France and Brazil. Which is more densely populated and by how much?"
)
print(f"\nFinal Answer: {result}")


--- Agent Step 1 ---


  Tool: search_knowledge({'query': 'population and area of France and Brazil population density France Brazil population area 2023'})
  Result: [{"country": "France", "category": "population", "value": "68 million"}, {"country": "Japan", "category": "population", "value": "125 million"}, {"country": "Brazil", "category": "population", "value": "214 million"}, {"country": "Germany", "category": "population", "value": "84 million"}, {"country": "France", "category": "capital", "value": "Paris"}, {"country": "Brazil", "category": "capital", "value": "Brasilia"}, {"country": "France", "category": "gdp", "value": "$2.78 trillion"}, {"country": "Brazil", "category": "gdp", "value": "$1.92 trillion"}, {"country": "France", "category": "area_km2", "value": 643801}, {"country": "Brazil", "category": "area_km2", "value": 8515767}]

--- Agent Step 2 ---


  Tool: calculator({'expression': '68000000 / 643801'})
  Result: {"expression": "68000000 / 643801", "result": 105.6227}

--- Agent Step 3 ---


  Tool: calculator({'expression': '214000000 / 8515767'})
  Result: {"expression": "214000000 / 8515767", "result": 25.129856}

--- Agent Step 4 ---


  Tool: calculator({'expression': '105.6227 - 25.129856'})
  Result: {"expression": "105.6227 - 25.129856", "result": 80.492844}

--- Agent Step 5 ---


Agent finished (no more tool calls)

Final Answer: - France: population ≈ 68,000,000; area = 643,801 km² → density ≈ $ \\frac{68{,}000{,}000}{643{,}801} \\approx 105.6$ people/km²  
- Brazil: population ≈ 214,000,000; area = 8,515,767 km² → density ≈ $ \\frac{214{,}000{,}000}{8{,}515{,}767} \\approx 25.13$ people/km²

Conclusion: France is more densely populated by about $105.6-25.13\\approx 80.49$ people/km² (≈4.2× denser).


## Summary

**Agent architecture:**
1. Receive user task
2. LLM decides what tool to use (or gives final answer)
3. Execute tool, feed result back to LLM
4. Repeat until done (or max iterations reached)

**Key takeaways:**
- Agents = LLM + Tools + Loop
- Always set a `max_iterations` to prevent infinite loops
- Good system prompts improve agent reasoning
- Conversation memory enables multi-turn interactions
- Complex tasks are broken into tool calls automatically by the LLM

**Next session:** Advanced patterns â€” chaining, guardrails, evaluation!